# 04 — GridSearchCV Tuning + Final Evaluation
Tune Ridge & Random Forest → test set metrics → residual analysis

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')

with open('../data/splits.pkl', 'rb') as f:
    X_train, X_test, y_train, y_test, NUM, ORD, NOM = pickle.load(f)

with open('../data/cv_results.pkl', 'rb') as f:
    cv_results, preprocessor = pickle.load(f)

print('Data loaded.')

## GridSearchCV — Ridge

In [ ]:
ridge_pipe = Pipeline([('pre', preprocessor), ('model', Ridge())])
ridge_grid = GridSearchCV(
    ridge_pipe,
    param_grid={'model__alpha': [0.1, 1, 5, 10, 50, 100, 200]},
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
)
ridge_grid.fit(X_train, y_train)

print(f'Best Ridge alpha : {ridge_grid.best_params_["model__alpha"]}')
print(f'Best CV RMSE     : {-ridge_grid.best_score_:.4f}')

## GridSearchCV — Random Forest

In [ ]:
rf_pipe = Pipeline([('pre', preprocessor), ('model', RandomForestRegressor(random_state=42, n_jobs=-1))])
rf_grid = GridSearchCV(
    rf_pipe,
    param_grid={
        'model__n_estimators': [100, 200],
        'model__max_depth':    [None, 15, 25],
        'model__min_samples_leaf': [1, 2],
    },
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
)
rf_grid.fit(X_train, y_train)

print(f'Best RF params : {rf_grid.best_params_}')
print(f'Best CV RMSE   : {-rf_grid.best_score_:.4f}')

## Test Set Evaluation

In [ ]:
def evaluate(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)
    print(f'{name:30s}  RMSE={rmse:.4f}  MAE={mae:.4f}  R²={r2:.4f}')
    return y_pred, rmse, mae, r2

ridge_pred, ridge_rmse, ridge_mae, ridge_r2 = evaluate('Ridge (tuned)',        ridge_grid.best_estimator_, X_test, y_test)
rf_pred,    rf_rmse,    rf_mae,    rf_r2    = evaluate('Random Forest (tuned)', rf_grid.best_estimator_,    X_test, y_test)

## Residual Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, pred, name in [
    (axes[0], ridge_pred, 'Ridge (tuned)'),
    (axes[1], rf_pred,    'Random Forest (tuned)'),
]:
    residuals = y_test.values - pred
    ax.scatter(pred, residuals, alpha=0.4, s=15, color='steelblue')
    ax.axhline(0, color='red', linewidth=1, linestyle='--')
    ax.set_xlabel('Predicted log(SalePrice)')
    ax.set_ylabel('Residual')
    ax.set_title(f'Residuals — {name}')

plt.tight_layout()
plt.savefig('../data/residuals.png', bbox_inches='tight')
plt.show()

## Predicted vs Actual

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, pred, name in [
    (axes[0], ridge_pred, 'Ridge (tuned)'),
    (axes[1], rf_pred,    'Random Forest (tuned)'),
]:
    ax.scatter(y_test, pred, alpha=0.4, s=15, color='steelblue')
    lims = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
    ax.plot(lims, lims, 'r--', linewidth=1)
    ax.set_xlabel('Actual log(SalePrice)')
    ax.set_ylabel('Predicted log(SalePrice)')
    ax.set_title(f'Predicted vs Actual — {name}')

plt.tight_layout()
plt.savefig('../data/predicted_vs_actual.png', bbox_inches='tight')
plt.show()

## Model Comparison Table

In [ ]:
summary = pd.DataFrame({
    'Model': ['Linear Regression', 'Ridge (alpha=10, CV)', 'Ridge (tuned, test)', 'Random Forest (tuned, test)'],
    'RMSE':  [
        cv_results['Linear Regression']['RMSE'],
        cv_results['Ridge (alpha=10)']['RMSE'],
        ridge_rmse,
        rf_rmse,
    ],
    'MAE': [
        cv_results['Linear Regression']['MAE'],
        cv_results['Ridge (alpha=10)']['MAE'],
        ridge_mae,
        rf_mae,
    ],
    'R²': [
        cv_results['Linear Regression']['R2'],
        cv_results['Ridge (alpha=10)']['R2'],
        ridge_r2,
        rf_r2,
    ],
}).set_index('Model')

print(summary.round(4).to_string())